In [ ]:
# 프로젝트 경로 설정
from pathlib import Path
import os
import sys

project_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "project_paths.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("프로젝트 폴더에서 노트북을 실행하세요.")
if str(project_root) in sys.path:
    sys.path.remove(str(project_root))
sys.path.insert(0, str(project_root))
sys.modules.pop("project_paths", None)
from project_paths import DATA_ROOT, RAW_DIR, PROCESSED_DIR, OUTPUT_DIR

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import RepeatedKFold, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
import shap
import matplotlib.pyplot as plt
from scipy.stats import scoreatpercentile
from tqdm import tqdm
import time

# 1. 파일 이름 정의
AMC_INPUT_PATH = str(PROCESSED_DIR / 'amc_tidy.csv')

amc_df = pd.read_csv(AMC_INPUT_PATH, encoding= 'utf-8', dtype='str')

# 1. 파일 이름 정의
SMC_INPUT_PATH = str(RAW_DIR / 'D2024_0087_02(op_id 포함).csv')

smc_df = pd.read_csv(SMC_INPUT_PATH, encoding= 'utf-8', dtype='str')
smc_df['row_no'] = smc_df['row_no'].astype('int')
smc_df = smc_df.sort_values('row_no').reset_index(drop=True)

def process_age(age_str):
    age_str = str(age_str).strip()
    if '이상' in age_str:
        return 80
    elif '-' in age_str:
        return 20
    elif '대' in age_str:
        return int(age_str.replace('대', ''))
    return 0


smc_df['age'] = smc_df['age'].apply(process_age).astype(int)
smc_df = smc_df.drop(['op_id'], axis =1)


In [ ]:
# smc_df 데이터 전처리
MIN_PROCEDURE_COUNT = 50
value_counts = smc_df['procedure_cd_fin'].value_counts()
frequent_codes = value_counts[value_counts >= MIN_PROCEDURE_COUNT].index
smc_df = smc_df[smc_df['procedure_cd_fin'].isin(frequent_codes)]

columns_to_int = ['age', 'use_quan','msbos']
target_int = int

columns_to_float = ['Hb', 'PT_inr','Plt','aPTT']
target_float = float

dtype_int = {col: target_int for col in columns_to_int}
smc_df = smc_df.astype(dtype_int)

dtype_float = {col: target_float for col in columns_to_float}
smc_df = smc_df.astype(dtype_float)

# ⭐ 사용할 컬럼 목록 정의 (20개)
MODEL_COLUMNS = [
    'anesthesia_cd', 'mi', 'pvd', 'surgeon_dept_cd', 'procedure_cd_fin',
    'mld', 'diabwc', 'rend', 'canc', 'Hb', 'PT_inr', 'Plt', 'aPTT',
    'heparins', 'coumarin_deriatives', 'use_quan' ,'msbos'
]
TARGET_COLUMN = 'use_quan'

smc_df = smc_df[MODEL_COLUMNS]

feature_columns = [c for c in MODEL_COLUMNS if c != TARGET_COLUMN]

feature_count = len(feature_columns)
print("--- 데이터 로드 및 전처리 완료 ---\n")


In [ ]:
# 1. interquartile_range 및 임계값 계산
# -------------------------------------------------------------
TARGET_COLUMN = 'use_quan'
MSBOS_COLUMN = 'msbos'

# use_quan이 0보다 큰 데이터만 사용하여 interquartile_range 계산 (수혈을 받은 환자 대상)
# 0을 포함하여 계산하면 1.5*interquartile_range 기준이 매우 낮아질 수 있습니다.
# 여기서는 일반적인 interquartile_range 기준 적용을 위해 0을 포함하여 계산합니다.
lower_quartile = smc_df[TARGET_COLUMN].quantile(0.25)
upper_quartile = smc_df[TARGET_COLUMN].quantile(0.75)
interquartile_range = upper_quartile - lower_quartile

# interquartile_range 상한 임계값 (Upper Bound)
iqr_upper_bound = upper_quartile + 1.5 * interquartile_range

print(f"use_quan Q1: {lower_quartile:.2f}")
print(f"use_quan Q3: {upper_quartile:.2f}")
print(f"use_quan IQR: {interquartile_range:.2f}")
print(f"use_quan IQR 상한 임계값 (Q3 + 1.5*IQR): {iqr_upper_bound:.2f}")

# -------------------------------------------------------------
# 2. 이상치 필터링 조건 생성
# -------------------------------------------------------------

# 조건 A: interquartile_range 상한을 초과하는 경우
exceeds_iqr_limit = (smc_df[TARGET_COLUMN] > iqr_upper_bound)

# 조건 B: MSBOS 값의 2배를 초과하는 경우
exceeds_msbos_limit = (smc_df[TARGET_COLUMN] > 2 * smc_df[MSBOS_COLUMN])

# 최종 이상치 조건: 조건 A와 조건 B를 동시에 만족
# np.logical_and를 사용하여 두 조건이 모두 참인 행을 찾습니다.
outlier_mask = np.logical_and(exceeds_iqr_limit, exceeds_msbos_limit)

# -------------------------------------------------------------
# 3. 이상치 제거
# -------------------------------------------------------------

# 이상치로 간주되는 행의 개수 확인
outlier_count = outlier_mask.sum()

# 이상치가 아닌 데이터만 남김 (필터링된 데이터프레임)
smc_filtered_df = smc_df[~outlier_mask].copy()

print(f"\n제거된 이상치 데이터 개수 (조건 A AND 조건 B 만족): {outlier_count} 개")
print(f"원래 데이터셋 크기: {len(smc_df)} 행")
print(f"이상치 제거 후 데이터셋 크기: {len(smc_filtered_df)} 행")
smc_filtered_df['institution'] = 'smc'
# 원본 df를 사용하지 않고 df_filtered를 사용하여 이후 모델링을 진행합니다.


In [ ]:
# amc_df 데이터 전처리
amc_df.columns
amc_df = amc_df.drop(['pt_id', 'op_id', 'anesthesia_cd', 'anesthesia_nm2'], axis=1).rename(columns = {'anesthesia_cd2':'anesthesia_cd',
                                                                                                             'use_quan_imputed':'use_quan'})

columns_to_int = ['age', 'use_quan','msbos']
target_int = int

columns_to_float = ['Hb', 'PT_inr','Plt','aPTT']
target_float = float

dtype_int = {col: target_int for col in columns_to_int}
amc_df = amc_df.astype(dtype_int)

dtype_float = {col: target_float for col in columns_to_float}
amc_df = amc_df.astype(dtype_float)

# ⭐ 사용할 컬럼 목록 정의 (20개)
MODEL_COLUMNS = [
    'anesthesia_cd', 'mi', 'pvd', 'surgeon_dept_cd', 'procedure_cd_fin',
    'mld', 'diabwc', 'rend', 'canc', 'Hb', 'PT_inr', 'Plt', 'aPTT',
    'heparins', 'coumarin_deriatives', 'use_quan' ,'msbos'
]
TARGET_COLUMN = 'use_quan'

amc_df = amc_df[MODEL_COLUMNS]

feature_columns = [c for c in MODEL_COLUMNS if c != TARGET_COLUMN]

feature_count = len(feature_columns)
print("--- 데이터 로드 및 전처리 완료 ---\n")
amc_df.info()


In [ ]:
# 1. interquartile_range 및 임계값 계산
# -------------------------------------------------------------
TARGET_COLUMN = 'use_quan'
MSBOS_COLUMN = 'msbos'

# use_quan이 0보다 큰 데이터만 사용하여 interquartile_range 계산 (수혈을 받은 환자 대상)
# 0을 포함하여 계산하면 1.5*interquartile_range 기준이 매우 낮아질 수 있습니다.
# 여기서는 일반적인 interquartile_range 기준 적용을 위해 0을 포함하여 계산합니다.
lower_quartile = amc_df[TARGET_COLUMN].quantile(0.25)
upper_quartile = amc_df[TARGET_COLUMN].quantile(0.75)
interquartile_range = upper_quartile - lower_quartile

# interquartile_range 상한 임계값 (Upper Bound)
iqr_upper_bound = upper_quartile + 1.5 * interquartile_range

print(f"use_quan Q1: {lower_quartile:.2f}")
print(f"use_quan Q3: {upper_quartile:.2f}")
print(f"use_quan IQR: {interquartile_range:.2f}")
print(f"use_quan IQR 상한 임계값 (Q3 + 1.5*IQR): {iqr_upper_bound:.2f}")

# -------------------------------------------------------------
# 2. 이상치 필터링 조건 생성
# -------------------------------------------------------------

# 조건 A: interquartile_range 상한을 초과하는 경우
exceeds_iqr_limit = (amc_df[TARGET_COLUMN] > iqr_upper_bound)

# 조건 B: MSBOS 값의 2배를 초과하는 경우
exceeds_msbos_limit = (amc_df[TARGET_COLUMN] > 2 * amc_df[MSBOS_COLUMN])

# 최종 이상치 조건: 조건 A와 조건 B를 동시에 만족
# np.logical_and를 사용하여 두 조건이 모두 참인 행을 찾습니다.
outlier_mask = np.logical_and(exceeds_iqr_limit, exceeds_msbos_limit)

# -------------------------------------------------------------
# 3. 이상치 제거
# -------------------------------------------------------------

# 이상치로 간주되는 행의 개수 확인
outlier_count = outlier_mask.sum()

# 이상치가 아닌 데이터만 남김 (필터링된 데이터프레임)
amc_filtered_df = amc_df[~outlier_mask].copy()

print(f"\n제거된 이상치 데이터 개수 (조건 A AND 조건 B 만족): {outlier_count} 개")
print(f"원래 데이터셋 크기: {len(amc_df)} 행")
print(f"이상치 제거 후 데이터셋 크기: {len(amc_filtered_df)} 행")
amc_filtered_df['institution'] = 'amc'
# 원본 df를 사용하지 않고 df_filtered를 사용하여 이후 모델링을 진행합니다.


In [ ]:
pd.concat([smc_filtered_df, amc_filtered_df]).reset_index(drop= 1).to_csv(str(OUTPUT_DIR / 'integration_data.csv'))
